# Week 21, Day 2: Linear equations with inputs: impulse responses and resonance

Companion notebook for the Engram lesson at https://engram-neuroai.vercel.app/learn/w021/d2.
Run the cells in order. Exercises have a checks cell after them: run it to test your answer.

### Exercise: Two ways to the same answer

Drive a membrane with τ = 15 ms by a random input current (RI changes every 1 ms, mean 5 mV, sd 8 mV) for 300 ms, with time step 0.01 ms. Compute the response by simulating τ dV/dt = −V + RI with small Euler steps (`V_sim`), and by convolving the input with h(t) = e^(−t/τ)/τ (`V_conv`, use np.convolve and multiply by dt).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
rng = np.random.default_rng(1967)
dt, T, tau = 0.01, 300.0, 15.0
t = np.arange(0, T, dt)
RI = np.repeat(5 + 8 * rng.normal(size=int(T)), int(1 / dt))[:len(t)]     # changes every 1 ms

V_sim = np.zeros(len(t))
for i in range(1, len(t)):
    V_sim[i] = ...

h = np.exp(-t / tau) / tau
V_conv = ...                                          # keep the first len(t) samples
print(f"largest difference {np.max(np.abs(V_conv - V_sim)):.4f} mV")
plt.plot(t, RI, alpha=0.3, label="input RI"); plt.plot(t, V_sim, label="simulated"); plt.plot(t, V_conv, "--", label="convolution"); plt.legend(); plt.xlabel("ms"); plt.show()

In [ ]:
import numpy as np
def test_agree():
    "the convolution and the direct simulation agree to within 0.05 mV everywhere"
    assert np.max(np.abs(V_conv - V_sim)) < 0.05

test_agree()
print("All checks passed")

### Exercise: From synaptic current to PSP

With τ_s = 2 ms and membrane τ = 15 ms, compute the alpha current on a 0.01 ms grid from 0 to 100 ms, convolve it with the membrane impulse response to get `psp`, and find the time of the PSP's peak, `t_peak_psp`.

In [ ]:
t = np.arange(0, 100, dt)
tau_s = 2.0
I_alpha = ...
h = np.exp(-t / tau) / tau
psp = np.convolve(I_alpha, h)[:len(t)] * dt
t_peak_psp = ...
print(f"current peaks at {t[np.argmax(I_alpha)]:.2f} ms, PSP peaks at {t_peak_psp:.2f} ms")
plt.plot(t, I_alpha / I_alpha.max(), label="synaptic current"); plt.plot(t, psp / psp.max(), label="PSP"); plt.legend(); plt.xlabel("ms"); plt.show()

In [ ]:
import numpy as np
def test_alpha():
    "the alpha current peaks at tau_s"
    assert abs(t[np.argmax(I_alpha)] - 2.0) < 0.02

def test_psp_later():
    "the PSP peaks later than the current, and decays more slowly (its value at 40 ms is more than 10% of its peak)"
    assert t_peak_psp > 2.0 + 2 and psp[np.searchsorted(t, 40)] > 0.1 * psp.max()

test_alpha()
test_psp_later()
print("All checks passed")

### Exercise: Find the resonance by simulation

With f0 = 8 Hz and ζ = 0.15, simulate x'' + 2ζω0 x' + ω0² x = cos(ωt) for drives at 2, 5, 8, 12 and 20 Hz, for 3 s with dt = 0.1 ms (write it as two first-order equations for x and v = x'). Measure the amplitude over the last second (`amp_sim`) and compare with the formula (`amp_formula`).

In [ ]:
w0, zeta = 2 * np.pi * 8, 0.15
freqs = [2, 5, 8, 12, 20]
dts, T2 = 1e-4, 3.0
amp_sim, amp_formula = [], []
for f in freqs:
    w = 2 * np.pi * f
    x, v = 0.0, 0.0
    n = int(T2 / dts); xs = np.empty(n)
    for i in range(n):
        a = ...                                   # acceleration from the equation
        x, v = x + dts * v, v + dts * a
        xs[i] = x
    amp_sim.append(np.abs(xs[-int(1 / dts):]).max())
    amp_formula.append(...)
amp_sim, amp_formula = np.array(amp_sim), np.array(amp_formula)
print("drive (Hz):", freqs); print("amplitude, simulated:", (amp_sim / amp_sim.max()).round(3)); print("amplitude, formula:  ", (amp_formula / amp_formula.max()).round(3))

In [ ]:
import numpy as np
def test_sim_matches():
    "the simulated steady-state amplitudes match the formula to within 5%"
    assert np.all(np.abs(amp_sim - amp_formula) / amp_formula < 0.05)

def test_peak():
    "the largest response is at the 8 Hz drive"
    assert freqs[np.argmax(amp_sim)] == 8

test_sim_matches()
test_peak()
print("All checks passed")